In [1]:
import torch
import os
import os 
import torch 
from PIL import Image
import geopandas as gpd
from IPython.display import display
from huggingface_hub import snapshot_download

from trident import OpenSlideWSI
from trident.segmentation_models import segmentation_model_factory
import h5py 

print(torch.cuda.is_available())
DEVICE = f"cuda:0" if torch.cuda.is_available() else "cpu"


from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

True


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# PATH = "../../.."
PATH = "storage"

local_wsi_dir = os.path.join(PATH, "data", "CAMELYON")

In [1]:
encoder_list = ["resnet50",
              "uni_v1", 
              "uni_v2", 
            #   "conch_v1", "conch_v15", 
              "virchow", 
              "virchow2", 
              "phikon", 
              "phikon_v2", 
              "gigapath", 
              "hoptimus0", 
              "hoptimus1", 
              # "musk", 
            #   "midnight12k", 
              # "kaiko-vits8", "kaiko-vits16", 
              # "kaiko-vitb8", "kaiko-vitb16", "kaiko-vitl14", 
              # "lunit-vits8", 
              "hibou_l", 
            #   "ctranspath"
            ]

len(encoder_list)

11

#### Segmentation and Extraction

In [4]:
import os
from pathlib import Path
import torch
import pandas as pd
import h5py
from trident import OpenSlideWSI
from trident.segmentation_models import segmentation_model_factory
from trident.patch_encoder_models import encoder_factory
import time

def extract_wsi_patch_features(
    encoder_name: str,
    target_mag: int,
    patch_size: int,
    wsi_dir: str,
    images: list[str],
    output_dir: str,
    mapping_csv: str,
    device: torch.device = None
) -> None:
    # 1) select device
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dev_idx = device.index if device.index is not None else 0
    device_str = f"{device.type}:{dev_idx}"

    # 2) load mapping
    mapping = pd.read_csv(mapping_csv, dtype={'sample_id': str})
    mapping.set_index('sample_id', inplace=True)

    # 3) prepare encoder
    encoder = encoder_factory(encoder_name)
    encoder.eval().to(device)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    for idx, fname in enumerate(images, start=1):
        total_start = time.time()
        seg_time = None
        feat_time = None

        sample_id = Path(fname).stem
        if sample_id not in mapping.index:
            raise KeyError(f"Sample ID '{sample_id}' not found in mapping CSV")
        slide_class = int(mapping.at[sample_id, 'status'])

        print(f"[{encoder_name}] ({idx}/{len(images)}) {fname} -> class={slide_class}")
        slide = OpenSlideWSI(slide_path=os.path.join(wsi_dir, fname), lazy_init=False)

        # --- segmentation (skip if GeoJSON already exists) ---
        geojson_dir = Path(output_dir) / "contours_geojson"
        geojson_dir.mkdir(exist_ok=True)
        geojson_path = geojson_dir / f"{sample_id}.geojson"
        seg_start = time.time()
        if not geojson_path.exists():
            print("  - segmenting tissue...")
            seg_model = segmentation_model_factory("hest")
            slide.segment_tissue(
                segmentation_model=seg_model,
                target_mag=target_mag,
                job_dir=output_dir,
                device=device_str
            )
            seg_time = time.time() - seg_start
        else:
            print("  - skipping segmentation (found)", geojson_path)
            seg_time = time.time() - seg_start

        # --- patch coords (skip if patches .h5 exists) ---
        patches_dir = Path(output_dir) / "patches"
        patches_dir.mkdir(exist_ok=True)
        coords_file = patches_dir / f"{sample_id}_patches.h5"
        if not coords_file.exists():
            print("  - extracting patch coordinates...")
            coords_file = slide.extract_tissue_coords(
                target_mag=target_mag,
                patch_size=patch_size,
                save_coords=output_dir
            )
        else:
            print("  - skipping coords extraction (found)", coords_file)

        # --- feature extraction (per-encoder dir) ---
        feats_dir = Path(output_dir) / f"features_{encoder_name}"
        feats_dir.mkdir(exist_ok=True)
        feats_file = feats_dir / f"{sample_id}.h5"
        feat_start = time.time()
        if not feats_file.exists():
            print("  - extracting patch features...")
            feats_file = slide.extract_patch_features(
                patch_encoder=encoder,
                coords_path=str(coords_file),
                save_features=str(feats_dir),
                device=device
            )
            feat_time = time.time() - feat_start
            with h5py.File(feats_file, 'a') as h5:
                h5.attrs['label'] = slide_class
            total_time = time.time() - total_start
            print(f"    -> saved features to {feats_file} | seg: {seg_time:.1f}s | feat: {feat_time:.1f}s | total: {total_time:.1f}s")
        else:
            feat_time = time.time() - feat_start
            total_time = time.time() - total_start
            print(f"  - skipping feature extraction (found) {feats_file} | seg: {seg_time:.1f}s | feat: {feat_time:.1f}s | total: {total_time:.1f}s")

        print()

In [5]:
import os

# ---------------------------------------------------------------------------- #
# 1) Your existing bin-packing function
# ---------------------------------------------------------------------------- #
def split_paths_by_size(paths: list[str], n_jobs: int) -> list[list[str]]:
    """
    Greedy “bin-packing” split of `paths` into `n_jobs` bins, balancing total file size.
    Returns a list of length n_jobs, each a list of paths.
    """
    ps = [(p, os.path.getsize(p)) for p in paths]
    ps.sort(key=lambda x: x[1], reverse=True)
    bins = [[] for _ in range(n_jobs)]
    bin_sizes = [0]*n_jobs
    for p, sz in ps:
        i = min(range(n_jobs), key=lambda j: bin_sizes[j])
        bins[i].append(p)
        bin_sizes[i] += sz
    return bins

# ---------------------------------------------------------------------------- #
# 2) Helper to split your image lists
# ---------------------------------------------------------------------------- #
def split_image_files_by_size(images: list[str], base_dir: str, n_jobs: int) -> list[list[str]]:
    """
    Given a list of filenames (e.g. 'test_001.tif'), constructs full paths
    under base_dir, splits into n_jobs bins by file size, then returns
    lists of filenames per bin.
    """
    full_paths = [os.path.join(base_dir, f) for f in images]
    bins = split_paths_by_size(full_paths, n_jobs)
    return [[os.path.basename(p) for p in bin] for bin in bins]

# ---------------------------------------------------------------------------- #
# 3) Collect your test/train image filenames
# ---------------------------------------------------------------------------- #
test_dir  = os.path.join(local_wsi_dir, "test")
train_dir = os.path.join(local_wsi_dir, "train")

test_images  = [f for f in os.listdir(test_dir)  if f.lower().endswith(".tif")]
train_images = [f for f in os.listdir(train_dir) if f.lower().endswith(".tif")]

# ---------------------------------------------------------------------------- #
# 4) Split into bins for parallel jobs
# ---------------------------------------------------------------------------- #
n_jobs = 4
test_bins  = split_image_files_by_size(test_images,  test_dir,  n_jobs)
train_bins = split_image_files_by_size(train_images, train_dir, n_jobs)

# Now test_bins[i] and train_bins[i] are the lists of filenames for job i
for i in range(n_jobs):
    print(f"Job {i} will process TEST slides:  {test_bins[i]}")
    print(f"Job {i} will process TRAIN slides: {train_bins[i]}\n")

# ---------------------------------------------------------------------------- #
# 5) Class-map paths (unchanged)
# ---------------------------------------------------------------------------- #
train_class_map = os.path.join(local_wsi_dir, "train_class_mapping.csv")
test_class_map  = os.path.join(local_wsi_dir, "test_class_mapping.csv")

Job 0 will process TEST slides:  ['test_006.tif']
Job 0 will process TRAIN slides: []

Job 1 will process TEST slides:  ['test_002.tif']
Job 1 will process TRAIN slides: []

Job 2 will process TEST slides:  ['test_007.tif', 'test_001.tif']
Job 2 will process TRAIN slides: []

Job 3 will process TEST slides:  ['test_008.tif', 'test_005.tif']
Job 3 will process TRAIN slides: []



In [ ]:
# print("--------------Beginning Training WSI extraction--------------")

# for enc in encoder_list:
#     extract_wsi_patch_features(
#         encoder_name=enc,
#         target_mag=2.5,
#         patch_size=512,
#         wsi_dir=train_dir,
#         images=train_bins[0],
#         output_dir=os.path.join(train_dir, "feats"),
#         mapping_csv=train_class_map
#     )

In [ ]:
print("--------------Beginning Test WSI extraction--------------")

for enc in encoder_list:
    extract_wsi_patch_features(
        encoder_name=enc,
        target_mag=2.5,
        patch_size=512,
        wsi_dir=test_dir,
        images=test_bins[3],
        output_dir=os.path.join(test_dir, "feats"),
        mapping_csv=test_class_map
    )

In [ ]:
# import h5py

# def inspect_h5_with_label(h5_path, print_values=False, max_items=5):
#     """
#     Inspect an HDF5 feature file and print:
#       - file-level attributes (including 'label' if present)
#       - for each dataset: name, shape, dtype, attributes
#       - optionally, a small sample of each dataset’s values
#     """
#     print(f"Inspecting: {h5_path}\n")
#     with h5py.File(h5_path, 'r') as f:
#         # 1) File‐level attrs
#         if f.attrs:
#             print("File-level attributes:")
#             for k, v in f.attrs.items():
#                 print(f"  {k}: {v}")
#         else:
#             print("No file-level attributes found.")
#         print()

#         # 2) Datasets and Groups
#         def _print_item(name, obj):
#             if isinstance(obj, h5py.Dataset):
#                 print(f"[Dataset] {name}")
#                 print(f"  shape: {obj.shape}, dtype: {obj.dtype}")
#                 if obj.attrs:
#                     print("  attributes:")
#                     for ak, av in obj.attrs.items():
#                         print(f"    {ak}: {av}")
#                 if print_values:
#                     try:
#                         data = obj[()]
#                         flat = data.ravel()
#                         preview = flat[:max_items]
#                         print(f"  sample values: {preview}")
#                     except Exception as e:
#                         print(f"  could not read values: {e}")
#                 print()
#             elif isinstance(obj, h5py.Group):
#                 print(f"[Group] {name}/")
#                 print()

#         f.visititems(_print_item)

# # Example usage:
# inspect_h5_with_label(
#     r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\feats\features_resnet50\test_001.h5",
#     print_values=True,
#     max_items=10
# )